# TMDB Movie Data Analysis

## Step 1: Fetching Raw Movie Data from the TMDB API

The raw movie data is fetched using The Movie Database (TMDB) API.  
The helper function defined in the `data_fetcher.py` is used for fecthing the day.

In [ ]:
pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

api_key = os.getenv("TMDB_API_KEY")

In [ ]:
print("API KEY:", os.getenv("TMDB_API_KEY"))

In [ ]:
# System setup to access external Python scripts
import sys
import os

sys.path.append(os.path.abspath('../scripts'))
import pandas as pd

# Import fetching function
from data_fetcher import fetch_tmdb_data

# Define API parameters
api_key = os.getenv("TMDB_API_KEY")
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397, 420818, 24428, 168259, 99861, 284054, 12445, 181808, 330457, 351286, 109445, 321612, 260513]

# Fetch raw data
df_raw = fetch_tmdb_data(movie_ids, api_key)

# Preview first few rows of the dataset
df_raw.head()

In [ ]:
print("Expected:", len(movie_ids))
print("Fetched:", len(df_raw))

In [ ]:
df_raw.to_csv("../data/raw_movies.csv", index=False)

# Step 2: Data Cleaning and Preprocessing

In [ ]:
# Import all data cleaning utilities
from data_cleaning import (
    parse_credits_column, extract_cast_crew_info, drop_irrelevant_columns,
    extract_all_fields, convert_column_datatypes, replace_unrealistic_values,
    convert_to_million, adjust_vote_average_by_genre, replace_placeholders,
    drop_duplicates_and_missing_ids_titles, drop_sparse_rows, filter_released_movies,
    reorder_columns, reset_dataframe_index
)

In [ ]:
# Make a copy of the raw data to clean
df_clean = df_raw.copy()

In [ ]:
# Handle 'credits' column
df_clean = parse_credits_column(df_clean)
df_cast_crew = df_clean['credits'].apply(extract_cast_crew_info)
df_clean = pd.concat([df_clean, df_cast_crew], axis=1)

In [ ]:
# Drop unnecessary columns
columns_to_drop = ['credits', 'video', 'adult', 'homepage', 'imdb_id', 'original_title', 'backdrop_path']
df_clean = drop_irrelevant_columns(df_clean, columns_to_drop)

In [ ]:
# Extract relevant fields from JSON columns
df_clean = extract_all_fields(df_clean)

In [ ]:
# Convert column data types
df_clean = convert_column_datatypes(df_clean)

In [ ]:
# Handle unrealistic numeric values
df_clean = replace_unrealistic_values(df_clean)

In [ ]:
# Convert budget & revenue to millions
df_clean = convert_to_million(df_clean)

In [ ]:
# Adjust vote_average for movies with vote_count == 0 
df_clean = adjust_vote_average_by_genre(df_clean)

In [ ]:
# Replace placeholder texts with NaN
df_clean = replace_placeholders(df_clean)

In [ ]:
# Drop duplicates and missing ids/titles
df_clean = drop_duplicates_and_missing_ids_titles(df_clean)

In [ ]:
# Drop sparse rows with too many missing values 
df_clean = drop_sparse_rows(df_clean, min_non_null=10)

In [ ]:
# Keep only released movies
df_clean = filter_released_movies(df_clean)

In [ ]:
# Reorder columns for final dataset
df_clean = reorder_columns(df_clean)

In [ ]:
# Reset index 
df_clean = reset_dataframe_index(df_clean)

In [ ]:
df_clean.head()

In [ ]:
df_clean.to_csv("../data/clean_movies.csv", index=False)

# Step 3: KPI Implementation & Analysis

In [ ]:
# Load the cleaned dataset
df_clean = pd.read_csv("../data/clean_movies.csv")

In [ ]:
from kpi_and_analysis import (
    calculate_profit, calculate_roi, kpi_ranking, filter_movies,
    franchise_vs_standalone, most_successful_franchises, most_successful_directors
)

In [ ]:
# Calculate KPI Rankings
kpi_results = kpi_ranking(df_clean)
print("KPI Rankings:", kpi_results)

In [ ]:
# Search Queries
# Search 1: Find the best-rated Science Fiction Action movies starring Bruce Willis
search_1 = filter_movies(df_clean, genre="Science Fiction|Action", actor="Bruce Willis", sort_by="vote_average")
print("\nBest-rated Science Fiction Action Movies starring Bruce Willis:", search_1)

In [ ]:
# Search 2: Find movies starring Uma Thurman, directed by Quentin Tarantino
search_2 = filter_movies(df_clean, actor="Uma Thurman", director="Quentin Tarantino", sort_by="runtime")
print("\nMovies starring Uma Thurman, directed by Quentin Tarantino:", search_2)

In [ ]:
# Franchise vs Standalone Movie Comparison
franchise_stats, standalone_stats = franchise_vs_standalone(df_clean)
print("\nFranchise Movie Stats:", franchise_stats)
print("Standalone Movie Stats:", standalone_stats)

In [ ]:
# Most Successful Franchises
franchise_success = most_successful_franchises(df_clean)
print("\nMost Successful Movie Franchises:", franchise_success)

In [ ]:
# Most Successful Directors
director_success = most_successful_directors(df_clean)
print("\nMost Successful Directors:", director_success)

# Step 4: Data Visualization

In [ ]:
import importlib
import visualizations
importlib.reload(visualizations)

from visualizations import plot_yearly_trends
from visualizations import plot_roi_distribution_by_genre

In [ ]:
# import all functions needed
from visualizations import (
    plot_revenue_vs_budget,
    plot_roi_distribution_by_genre,
    plot_popularity_vs_rating,
    plot_yearly_trends,
    plot_franchise_vs_standalone,
)

# Ensure this function is still available in data_cleaning.py if needed
from data_cleaning import convert_to_million

In [ ]:
# Load the clean dataset
df_clean = pd.read_csv("../data/clean_movies.csv")

In [ ]:
from visualizations import plot_revenue_vs_budget

print("Plotting Revenue vs Budget Trends...")
plot_revenue_vs_budget(df_clean)

In [ ]:
print("Plotting ROI Distribution by Genre...")
plot_roi_distribution_by_genre(df_clean)

In [ ]:
# Plot Popularity vs Rating
print("Plotting Popularity vs Rating...")
plot_popularity_vs_rating(df_clean)

In [ ]:
# Plot Yearly Trends in Box Office Performance
print("Plotting Yearly Trends in Box Office Performance...")
plot_yearly_trends(df_clean)

In [ ]:
# Create 'is_franchise' column from 'belongs_to_collection'
df_clean['is_franchise'] = df_clean['belongs_to_collection'].notnull().astype(int)

In [ ]:
# Plot Franchise vs Standalone Success
print("Plotting Franchise vs Standalone Success...")
plot_franchise_vs_standalone(df_clean)